# 09 Expanded Application Feature Groups

Goal: test feature groups suggested by the profiling JSON and feature-importance analysis. The notebook starts from the current curated engineered feature set, then adds contract/context, time-history, region/timing, address mismatch, bureau request, social-circle, and building summary groups. It keeps previous notebooks unchanged.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src.features import BASIC_APPLICATION_FEATURES, create_features
from src.preprocessing import build_preprocessor
from src.thresholding import (
    evaluate_probabilities,
    find_best_threshold,
    get_positive_probabilities,
)


In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.2

LIGHTGBM_PARAMS = {
    "subsample": 1.0,
    "reg_lambda": 0.0,
    "num_leaves": 31,
    "n_estimators": 600,
    "min_child_samples": 50,
    "learning_rate": 0.02,
    "colsample_bytree": 0.85,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
    "importance_type": "gain",
}

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
train_df = pd.read_csv(RAW_DATA_DIR / "application_train.csv")
y = train_df["TARGET"].copy()

train_index, test_index = train_test_split(
    train_df.index,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

train_full_index, valid_index = train_test_split(
    train_index,
    test_size=VALIDATION_SIZE,
    stratify=y.loc[train_index],
    random_state=RANDOM_STATE,
)

y_train = y.loc[train_full_index]
y_valid = y.loc[valid_index]
y_train_full = y.loc[train_index]
y_test = y.loc[test_index]

train_df.shape, y_train.mean(), y_valid.mean(), y_test.mean()


((307511, 122),
 np.float64(0.08072924605957135),
 np.float64(0.0807284256737531),
 np.float64(0.08072776937710356))

In [3]:
feature_groups = {
    "baseline_curated": BASIC_APPLICATION_FEATURES,
    "contract_context": [
        "NAME_CONTRACT_TYPE",
        "NAME_TYPE_SUITE",
        "FLAG_OWN_CAR",
        "FLAG_OWN_REALTY",
        "ORGANIZATION_TYPE",
    ],
    "time_history": [
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH",
        "DAYS_LAST_PHONE_CHANGE",
    ],
    "region_timing": [
        "REGION_POPULATION_RELATIVE",
        "REGION_RATING_CLIENT",
        "REGION_RATING_CLIENT_W_CITY",
        "WEEKDAY_APPR_PROCESS_START",
        "HOUR_APPR_PROCESS_START",
    ],
    "address_mismatch": [
        "REG_REGION_NOT_LIVE_REGION",
        "REG_REGION_NOT_WORK_REGION",
        "LIVE_REGION_NOT_WORK_REGION",
        "REG_CITY_NOT_LIVE_CITY",
        "REG_CITY_NOT_WORK_CITY",
        "LIVE_CITY_NOT_WORK_CITY",
    ],
    "bureau_requests": [
        "AMT_REQ_CREDIT_BUREAU_HOUR",
        "AMT_REQ_CREDIT_BUREAU_DAY",
        "AMT_REQ_CREDIT_BUREAU_WEEK",
        "AMT_REQ_CREDIT_BUREAU_MON",
        "AMT_REQ_CREDIT_BUREAU_QRT",
        "AMT_REQ_CREDIT_BUREAU_YEAR",
    ],
    "social_circle": [
        "OBS_30_CNT_SOCIAL_CIRCLE",
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "OBS_60_CNT_SOCIAL_CIRCLE",
        "DEF_60_CNT_SOCIAL_CIRCLE",
    ],
    "building_summaries": [
        column
        for column in train_df.columns
        if column.endswith(("_AVG", "_MODE", "_MEDI"))
        and pd.api.types.is_numeric_dtype(train_df[column])
    ],
    "possible_related_extra": [
        "OWN_CAR_AGE",
        "TOTALAREA_MODE",
        "FLAG_EMP_PHONE",
        "FLAG_WORK_PHONE",
        "FLAG_PHONE",
        "FLAG_EMAIL",
    ],
}

{
    group_name: len([column for column in columns if column in train_df.columns])
    for group_name, columns in feature_groups.items()
}


{'baseline_curated': 17,
 'contract_context': 5,
 'time_history': 3,
 'region_timing': 5,
 'address_mismatch': 6,
 'bureau_requests': 6,
 'social_circle': 4,
 'building_summaries': 43,
 'possible_related_extra': 6}

In [4]:
def unique_existing_columns(columns):
    seen = set()
    selected = []
    for column in columns:
        if column in train_df.columns and column not in seen:
            selected.append(column)
            seen.add(column)
    return selected


def build_feature_frame(raw_columns):
    raw_columns = unique_existing_columns(raw_columns)
    return create_features(train_df[raw_columns].copy())


def build_lgbm_pipeline(X_train):
    numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
    categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

    preprocessor = build_preprocessor(
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        one_hot_sparse_output=True,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LGBMClassifier(**LIGHTGBM_PARAMS)),
        ]
    )


def evaluate_raw_feature_set(name, raw_columns):
    X_full = build_feature_frame(raw_columns)
    X_train = X_full.loc[train_full_index]
    X_valid = X_full.loc[valid_index]

    pipeline = build_lgbm_pipeline(X_train)
    pipeline.fit(X_train, y_train)

    valid_probabilities = get_positive_probabilities(pipeline, X_valid)
    threshold_info = find_best_threshold(y_valid, valid_probabilities)
    valid_metrics = evaluate_probabilities(
        y_true=y_valid,
        probabilities=valid_probabilities,
        threshold=threshold_info["threshold"],
    )

    return {
        "feature_set": name,
        "raw_feature_count": len(unique_existing_columns(raw_columns)),
        "engineered_feature_count": X_full.shape[1],
        "threshold": threshold_info["threshold"],
        "validation_roc_auc": valid_metrics["roc_auc"],
        "validation_average_precision": valid_metrics["average_precision"],
        "validation_precision_class_1": valid_metrics["precision_class_1"],
        "validation_recall_class_1": valid_metrics["recall_class_1"],
        "validation_f1_class_1": valid_metrics["f1_class_1"],
        "raw_columns": unique_existing_columns(raw_columns),
        "engineered_columns": X_full.columns.tolist(),
    }


In [5]:
baseline_raw_columns = feature_groups["baseline_curated"]

incremental_specs = []
current_columns = list(baseline_raw_columns)

incremental_specs.append(("baseline_curated_engineered", current_columns.copy()))

for group_name in [
    "contract_context",
    "time_history",
    "region_timing",
    "address_mismatch",
    "bureau_requests",
    "social_circle",
    "building_summaries",
    "possible_related_extra",
]:
    current_columns = unique_existing_columns(
        current_columns + feature_groups[group_name]
    )
    incremental_specs.append((f"add_through_{group_name}", current_columns.copy()))

single_group_specs = [
    (
        f"baseline_plus_{group_name}",
        unique_existing_columns(baseline_raw_columns + feature_groups[group_name]),
    )
    for group_name in [
        "contract_context",
        "time_history",
        "region_timing",
        "address_mismatch",
        "bureau_requests",
        "social_circle",
        "building_summaries",
        "possible_related_extra",
    ]
]

len(incremental_specs), len(single_group_specs)


(9, 8)

In [6]:
experiment_results = []

for name, columns in incremental_specs + single_group_specs:
    print(f"Training: {name} ({len(columns)} raw columns)")
    result = evaluate_raw_feature_set(name, columns)
    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results)
results_display_df = results_df.drop(columns=["raw_columns", "engineered_columns"])
results_display_df = results_display_df.sort_values(
    ["validation_roc_auc", "validation_f1_class_1"],
    ascending=False,
).reset_index(drop=True)

results_display_df


Training: baseline_curated_engineered (17 raw columns)


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training: add_through_contract_context (22 raw columns)


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training: add_through_time_history (25 raw columns)


...

Training: baseline_plus_building_summaries (60 raw columns)


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training: baseline_plus_possible_related_extra (23 raw columns)


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,feature_set,raw_feature_count,engineered_feature_count,threshold,validation_roc_auc,validation_average_precision,validation_precision_class_1,validation_recall_class_1,validation_f1_class_1
0,add_through_possible_related_extra,94,128,0.688844,0.762402,0.243919,0.257637,0.382175,0.307786
1,add_through_social_circle,46,77,0.679823,0.761799,0.243105,0.250158,0.397533,0.307079
2,add_through_bureau_requests,42,70,0.671914,0.761662,0.243210,0.243506,0.410624,0.305717
3,add_through_building_summaries,89,123,0.678201,0.761566,0.243647,0.249490,0.400050,0.307320
4,add_through_address_mismatch,36,61,0.677335,0.761148,0.242680,0.247785,0.401309,0.306391
5,add_through_region_timing,30,54,0.687171,0.761005,0.241481,0.253850,0.385952,0.306263
6,baseline_plus_possible_related_extra,23,41,0.654929,0.760422,0.241821,0.235364,0.446375,0.308214
7,baseline_plus_region_timing,22,40,0.674340,0.760184,0.243133,0.244258,0.409617,0.306028
8,add_through_contract_context,22,37,0.678138,0.759924,0.240915,0.247568,0.403575,0.306882
9,baseline_plus_contract_context,22,37,0.678138,0.759924,0.240915,0.247568,0.403575,0.306882


In [7]:
best_row = results_df.loc[
    results_df["validation_roc_auc"].idxmax()
]
best_feature_set = best_row["feature_set"]
best_raw_columns = best_row["raw_columns"]
best_engineered_columns = best_row["engineered_columns"]
best_threshold = best_row["threshold"]

best_feature_set, len(best_raw_columns), len(best_engineered_columns), best_threshold


('add_through_possible_related_extra', 94, 128, np.float64(0.6888442787524717))

In [8]:
best_raw_columns


['AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'CNT_CHILDREN',
 'CNT_FAM_MEMBERS',
 'CODE_GENDER',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_INCOME_TYPE',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'NAME_CONTRACT_TYPE',
 'NAME_TYPE_SUITE',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'ORGANIZATION_TYPE',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'DAYS_LAST_PHONE_CHANGE',
 'REGION_POPULATION_RELATIVE',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'OB

In [9]:
best_engineered_columns


['AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'DAYS_EMPLOYED',
 'CNT_CHILDREN',
 'CNT_FAM_MEMBERS',
 'CODE_GENDER',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_INCOME_TYPE',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'NAME_CONTRACT_TYPE',
 'NAME_TYPE_SUITE',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'ORGANIZATION_TYPE',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'DAYS_LAST_PHONE_CHANGE',
 'REGION_POPULATION_RELATIVE',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'OBS_30_CNT_SOCIAL

In [10]:
X_best_full = build_feature_frame(best_raw_columns)
X_train_full_best = X_best_full.loc[train_index]
X_test_best = X_best_full.loc[test_index]

final_pipeline = build_lgbm_pipeline(X_train_full_best)
final_pipeline.fit(X_train_full_best, y_train_full)

test_probabilities = get_positive_probabilities(final_pipeline, X_test_best)

final_default_metrics = evaluate_probabilities(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=0.5,
)

final_selected_metrics = evaluate_probabilities(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=best_threshold,
)

final_test_df = pd.DataFrame([
    {
        "feature_set": best_feature_set,
        "threshold_strategy": "default_0.5",
        **final_default_metrics,
    },
    {
        "feature_set": best_feature_set,
        "threshold_strategy": "validation_selected",
        **final_selected_metrics,
    },
])

final_test_df


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,feature_set,threshold_strategy,threshold,roc_auc,average_precision,accuracy,precision_class_1,recall_class_1,f1_class_1
0,add_through_possible_related_extra,default_0.5,0.500000,0.769069,0.258392,0.714876,0.174714,0.679960,0.277997
1,add_through_possible_related_extra,validation_selected,0.688844,0.769069,0.258392,0.864982,0.268735,0.390735,0.318450


In [11]:
for threshold_strategy, threshold in [
    ("default_0.5", 0.5),
    ("validation_selected", best_threshold),
]:
    predictions = (test_probabilities >= threshold).astype(int)

    print(threshold_strategy)
    print("threshold:", threshold)
    print(confusion_matrix(y_test, predictions))
    print(classification_report(y_test, predictions, zero_division=0))


default_0.5
threshold: 0.5
[[40591 15947]
 [ 1589  3376]]
              precision    recall  f1-score   support

           0       0.96      0.72      0.82     56538
           1       0.17      0.68      0.28      4965

    accuracy                           0.71     61503
   macro avg       0.57      0.70      0.55     61503
weighted avg       0.90      0.71      0.78     61503

validation_selected
threshold: 0.6888442787524717
[[51259  5279]
 [ 3025  1940]]


              precision    recall  f1-score   support

           0       0.94      0.91      0.93     56538
           1       0.27      0.39      0.32      4965

    accuracy                           0.86     61503
   macro avg       0.61      0.65      0.62     61503
weighted avg       0.89      0.86      0.88     61503



In [12]:
transformed_feature_names = final_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

gain_importance = final_pipeline.named_steps[
    "model"
].feature_importances_

feature_importance_df = pd.DataFrame({
    "transformed_feature": transformed_feature_names,
    "gain_importance": gain_importance,
}).sort_values("gain_importance", ascending=False)

feature_importance_df.head(40)


,transformed_feature,gain_importance
98,numeric__EXT_SOURCE_MEAN,666156.813121
93,numeric__CREDIT_TERM_APPROX,151115.401924
99,numeric__EXT_SOURCE_MIN,144628.518682
100,numeric__EXT_SOURCE_MAX,131784.155112
92,numeric__GOODS_CREDIT_RATIO,59680.905898
9,numeric__EXT_SOURCE_3,55006.281667
81,numeric__AGE_YEARS,54098.198538
2,numeric__AMT_ANNUITY,29703.781933
83,numeric__EMPLOYMENT_AGE_RATIO,26937.778247
3,numeric__AMT_GOODS_PRICE,26659.744905


## Summary

The tables above show which added groups improve validation ROC-AUC and F1. The final holdout metrics are reported only for the best validation-ROC-AUC feature set.
